In [ ]:
import re
import pandas as pd
from pathlib import Path

def parse_trial_history(file_path):
    """
    Parse the Full Trial History table from an Experiment_Report.md file.
    
    Parameters
    ----------
    file_path : str or Path
        Path to the Experiment_Report.md file
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: Run, Model Description, Error, Integrity, 
        Accuracy, Details, Optimality, Total
        
    Examples
    --------
    >>> Files = {
    ...     'Burgers': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md'
    ... }
    >>> df = parse_trial_history(Files['Burgers'])
    >>> # Plot error vs run
    >>> import matplotlib.pyplot as plt
    >>> plt.semilogy(df['Run'], df['Error'])
    >>> plt.xlabel('Run')
    >>> plt.ylabel('Error')
    >>> # Extract scores
    >>> print(df[['Run', 'Total']])
    """
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find the Full Trial History section
    history_pattern = r'## Full Trial History\s*\n(.*?)(?=\n##|\n###|$)'
    match = re.search(history_pattern, content, re.DOTALL)
    
    if not match:
        raise ValueError("Could not find 'Full Trial History' section in file")
    
    history_section = match.group(1)
    
    # Split into lines and find the table
    lines = history_section.strip().split('\n')
    
    # Find header and separator lines
    header_idx = None
    sep_idx = None
    for i, line in enumerate(lines):
        if '| Run |' in line or '| :--- |' in line:
            if header_idx is None and '| Run |' in line:
                header_idx = i
            elif sep_idx is None and '| :--- |' in line:
                sep_idx = i
                break
    
    if header_idx is None or sep_idx is None:
        raise ValueError("Could not find table header in Full Trial History")
    
    # Parse header
    header_line = lines[header_idx]
    headers = [h.strip().replace('**', '') for h in header_line.split('|')[1:-1]]
    
    # Parse data rows (everything after separator)
    data_rows = []
    for line in lines[sep_idx + 1:]:
        line = line.strip()
        if not line or not line.startswith('|'):
            break
        
        # Split by | and clean up, removing markdown bold formatting
        cells = [cell.strip().replace('**', '') for cell in line.split('|')[1:-1]]
        if len(cells) == len(headers):
            data_rows.append(cells)
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=headers)
    
    # Convert numeric columns
    df['Run'] = df['Run'].astype(int) + 1  # Add 1 to all run numbers since files start at 0
    
    # Convert Error column (handles scientific notation like 1.00e-06, percentages like 0.02%, and N/A)
    def parse_error(x):
        x = x.strip()
        if not x or x.upper() == 'N/A' or x.lower() == 'not applicable':
            return 0.0
        # Handle percentage values
        if '%' in x:
            return float(x.replace('%', '')) / 100.0
        # Handle scientific notation and regular floats
        return float(x)
    
    df['Error'] = df['Error'].apply(parse_error)
    
    # Convert score columns
    score_cols = ['Integrity', 'Accuracy', 'Details', 'Optimality', 'Total']
    for col in score_cols:
        df[col] = df[col].astype(int)
    
    return df


def extract_metrics(df, metric_col):
    """
    Extract a specific metric column for easy plotting.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame returned by parse_trial_history
    metric_col : str
        Column name to extract (e.g., 'Error', 'Total', 'Accuracy')
        
    Returns
    -------
    runs : array
        Run numbers
    values : array
        Metric values
    """
    return df['Run'].values, df[metric_col].values


def plot_error_progression(df, ax=None):
    """
    Quick plot of error vs run number.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame from parse_trial_history
    ax : matplotlib axis, optional
        Axis to plot on. If None, creates new figure.
        
    Returns
    -------
    ax : matplotlib axis
    """
    import matplotlib.pyplot as plt
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.semilogy(df['Run'], df['Error'], 'o-', linewidth=2, markersize=8)
    ax.set_xlabel('Run', fontsize=12)
    ax.set_ylabel('Relative L2 Error', fontsize=12)
    ax.set_title('Error Progression Across Runs', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_score_breakdown(df, ax=None):
    """
    Stacked bar chart showing score breakdown by component.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame from parse_trial_history
    ax : matplotlib axis, optional
        Axis to plot on. If None, creates new figure.
        
    Returns
    -------
    ax : matplotlib axis
    """
    import matplotlib.pyplot as plt
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    
    score_cols = ['Integrity', 'Accuracy', 'Details', 'Optimality']
    
    bottom = None
    for col in score_cols:
        if bottom is None:
            ax.bar(df['Run'], df[col], label=col)
            bottom = df[col].values
        else:
            ax.bar(df['Run'], df[col], bottom=bottom, label=col)
            bottom = bottom + df[col].values
    
    ax.set_xlabel('Run', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Score Breakdown Across Runs', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    return ax


# Example usage
if __name__ == "__main__":
    # Example with your file structure
    Files = {
        'Allen_Cahn': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
        'Burgers': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
        'Helmholtz': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
        'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
        'Inviscid Burgers': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md'
    }
    
    # Parse all histories
    dfs = {}
    for name, path in Files.items():
        try:
            dfs[name] = parse_trial_history(path)
            print(f"✓ Loaded {name}: {len(dfs[name])} runs")
        except Exception as e:
            print(f"✗ Failed to load {name}: {e}")
    
    # Print summary statistics for each problem
    print("\n" + "="*80)
    print("SUMMARY STATISTICS")
    print("="*80)
    for name, df in dfs.items():
        print(f"\n{name}:")
        print(f"  Number of runs: {len(df)}")
        print(f"  Best error: {df['Error'].min():.2e} (Run {df.loc[df['Error'].idxmin(), 'Run']})")
        print(f"  Best score: {df['Total'].max()} (Run {df.loc[df['Total'].idxmax(), 'Run']})")
        print(f"  Final score: {df['Total'].iloc[-1]} (Run {df['Run'].iloc[-1]})")
    
    # Plot comparison of Total score vs iteration
    try:
        import matplotlib.pyplot as plt
        import matplotlib.cm as cm
        import numpy as np
        
        fig, ax = plt.subplots(figsize=(14, 8))
        
        # Automatically generate colors and markers
        n_problems = len(dfs)
        colors = cm.tab10(np.linspace(0, 1, n_problems))
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
        
        # Plot all problems with Run=0, Score=0 baseline
        for idx, (name, df) in enumerate(dfs.items()):
            # Add Run=0, Score=0 as starting point
            runs = [0] + df['Run'].tolist()
            scores = [0] + df['Total'].tolist()
            
            ax.plot(runs, scores, 
                   marker=markers[idx % len(markers)], 
                   color=colors[idx],
                   linewidth=2.5, 
                   markersize=10,
                   label=name.replace('_', ' '),
                   alpha=0.8)
        
        ax.set_xlabel('Run / Iteration', fontsize=14, fontweight='bold')
        ax.set_ylabel('Total Score', fontsize=14, fontweight='bold')
        ax.set_title('ATHENA Performance: Total Score Evolution Across Problems', 
                    fontsize=16, fontweight='bold', pad=20)
        ax.legend(fontsize=12, framealpha=0.95, loc='lower right')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_ylim([0, 105])
        
        # Add horizontal line at score=95 (typical threshold for good performance)
        ax.axhline(y=95, color='gray', linestyle='--', linewidth=1, alpha=0.5)
        
        plt.tight_layout()
        plt.savefig('total_score_comparison.png', dpi=300, bbox_inches='tight')
        print("\n✓ Plot saved to: total_score_comparison.png")
        
    except ImportError:
        print("\n✗ Matplotlib not available for plotting")

In [ ]:
# ============================================================================
# 1x2 PLOT: SciML vs Numerical Methods
# ============================================================================

Files_SciML = {
    'AC': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'B': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md'
}

Files_Numerical = {
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2025_12_08_0014/Experiment_Report.md',
    'KH': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT(cNS)': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}


Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_0937/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# Parse both sets
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}

# Create 1x2 subplot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

# LEFT: SciML
colors_sciml = cm.tab10(np.linspace(0, 1, len(dfs_sciml)))
for idx, (name, df) in enumerate(dfs_sciml.items()):
    runs = [0] + df['Run'].tolist()
    scores = [0] + df['Total'].tolist()
    ax1.plot(scores, marker=markers[idx % len(markers)], color=colors_sciml[idx],
             linewidth=2.5, markersize=10, label=name, alpha=0.8)

ax1.set_xlabel('$A_n$', fontsize=14, fontweight='bold')
ax1.set_ylabel('Reward', fontsize=14, fontweight='bold')
ax1.set_title('SciML (PINNs)', fontsize=16, fontweight='bold', pad=20)
ax1.legend(fontsize=11, framealpha=0.95, loc='lower right')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_ylim([0, 105])
ax1.axhline(y=95, color='k', linestyle='--', linewidth=1, alpha=1)

# RIGHT: Numerical
colors_numerical = cm.tab10(np.linspace(0, 1, len(dfs_numerical)))
for idx, (name, df) in enumerate(dfs_numerical.items()):
    runs = [0] + df['Run'].tolist()
    scores = [0] + df['Total'].tolist()
    ax2.plot(scores, marker=markers[idx % len(markers)], color=colors_numerical[idx],
             linewidth=2.5, markersize=10, label=name, alpha=0.8)

ax2.set_xlabel('$A_n$', fontsize=14, fontweight='bold')
ax2.set_ylabel('Reward', fontsize=14, fontweight='bold')
ax2.set_title('Numerical Methods', fontsize=16, fontweight='bold', pad=20)
ax2.legend(fontsize=11, framealpha=0.95, loc='lower right')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.set_ylim([0, 105])
ax2.axhline(y=95, color='k', linestyle='--', linewidth=1, alpha=1)

fig.suptitle('ATHENA Performance: SciML vs Numerical Methods', fontsize=18, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# ============================================================================
# FONTSIZE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16  # Change this value to adjust all fonts proportionally
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 6
TICK_FONTSIZE = BASE_FONTSIZE - 4

# Set font properties
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# 1x4 PLOT: SciML vs Numerical Methods + Analysis
# ============================================================================

Files_SciML_analytic = {
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
    'VB1D': '../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1605/Experiment_Report.md',
}

Files_SciML = {
    'AC': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'B': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md'
}

Files_Numerical = {
    'VB1D': '../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1554/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2025_12_08_0014/Experiment_Report.md',
    'KH': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'cNS': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}

Analysis_IB2D = {
    'IB1': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248/Experiment_Report.md',
    'IB2': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2303/Experiment_Report.md',
    'IB3': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1053/Experiment_Report.md',
    'IB4': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/Experiment_Report.md',
    'IB5': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
}

Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_0937/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# Parse all sets
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}
dfs_rt = {name: parse_trial_history(path) for name, path in Analysis_RT.items()}
dfs_hm = {name: parse_trial_history(path) for name, path in Analysis_HM.items()}

# Create 1x4 subplot
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

# Predefined colors (max 6 elements)
colors = ['black', 'red', 'blue', 'green', 'orange', 'cyan']

# Function to plot data
def plot_panel(ax, dfs, title, colors, y_label=False):
    print(ax)
    for idx, (name, df) in enumerate(dfs.items()):
        runs = [0] + df['Run'].tolist()
        scores = [0] + df['Total'].tolist()
        ax.plot(runs, scores, marker=markers[idx % len(markers)], 
                color=colors[idx % len(colors)],
                linewidth=1.5, markersize=5, label=name, alpha=0.8)
    
    ax.set_xlabel('$n$', fontsize=LABEL_FONTSIZE)
    if y_label:
        ax.set_ylabel('$R_n$', fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, ncols=2, loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim([0, 105])
    ax.axhline(y=95, color='k', linestyle='--', linewidth=1, alpha=1)
    ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot each panel
plot_panel(axes[1], dfs_sciml, 'SciML (All)', colors)
plot_panel(axes[0], dfs_numerical, 'SciC (All)', colors, y_label=True)
plot_panel(axes[2], dfs_rt, 'SciML (RT)', colors)
plot_panel(axes[3], dfs_hm, 'SciC (HM)', colors)

plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:

def parse_deeponet_trial_history(file_path):
    """
    Parse the Full Trial History table from a DeepONet Experiment_Report.md file.
    
    Parameters
    ----------
    file_path : str or Path
        Path to the DeepONet Experiment_Report.md file
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: Run, Model, Error
        
    Examples
    --------
    >>> deeponet_file = '/path/to/DeepONet_codes/.../Experiment_Report.md'
    >>> df = parse_deeponet_trial_history(deeponet_file)
    >>> # Plot error vs run
    >>> import matplotlib.pyplot as plt
    >>> plt.semilogy(df['Run'], df['Error'])
    >>> plt.xlabel('Run')
    >>> plt.ylabel('Error (%)')
    """
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find the Full Trial History section
    history_pattern = r'## Full Trial History\s*\n(.*?)(?=\n##|\Z)'
    match = re.search(history_pattern, content, re.DOTALL)
    
    if not match:
        raise ValueError("Could not find 'Full Trial History' section in file")
    
    history_section = match.group(1)
    
    # Split into lines
    lines = history_section.strip().split('\n')
    
    # Find header line (should contain "Run | Model | Error")
    header_idx = None
    sep_idx = None
    for i, line in enumerate(lines):
        if '| Model |' in line or 'Run | Model | Error' in line:
            header_idx = i
        elif '---' in line and '|' in line:
            sep_idx = i
            break
    
    if header_idx is None or sep_idx is None:
        raise ValueError("Could not find table header in DeepONet Full Trial History")
    
    # Parse data rows (everything after separator)
    data_rows = []
    for line in lines[sep_idx + 1:]:
        line = line.strip()
        if not line or not line.startswith(('|', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9')):
            # Stop if we hit empty line or non-data line
            if not line:
                continue
            if not any(c.isdigit() for c in line[:5]):
                break
        
        # Split by | and clean up
        if '|' in line:
            cells = [cell.strip() for cell in line.split('|')]
            # Remove empty first/last elements if line starts/ends with |
            cells = [c for c in cells if c]
            
            if len(cells) >= 3:
                data_rows.append(cells[:3])  # Take only first 3 columns (Run, Model, Error)
    
    if not data_rows:
        raise ValueError("No data rows found in DeepONet trial history table")
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=['Run', 'Model', 'Error'])
    
    # Convert Run to integer
    df['Run'] = df['Run'].astype(int)
    
    # Convert Error from percentage string to float (as percentage, not decimal)
    def parse_error_percent(x):
        x = x.strip()
        if not x or x.upper() == 'N/A':
            return np.nan
        # Remove % sign and convert to float
        if '%' in x:
            return float(x.replace('%', ''))
        return float(x)
    
    df['Error'] = df['Error'].apply(parse_error_percent)
    
    return df


# Example usage
if __name__ == "__main__":
    # Path to DeepONet experiment report
    deeponet_file = '../Results/DeepONet_codes/Burgers_Periodic_ICs/2025_11_09_1823_plots/Experiment_Report.md'
    
    # Parse the DeepONet trial history
    df_deeponet = parse_deeponet_trial_history(deeponet_file)
    
    print(f"✓ Loaded DeepONet results: {len(df_deeponet)} runs")
    print("\nDataFrame preview:")
    print(df_deeponet[['Run', 'Error']])
    
    # Extract error values
    errors = df_deeponet['Error'].values
    runs = df_deeponet['Run'].values  # Use actual run numbers from file
    
    # Plot
    import matplotlib.pyplot as plt
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.semilogy(runs, errors, 'o-', linewidth=2, markersize=8, color='blue')
    ax.set_xlabel('Run', fontsize=14)
    ax.set_ylabel('Relative L2 Error (%)', fontsize=14)
    ax.set_title('DeepONet: Error Progression Across Runs', fontsize=16)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig('deeponet_error_progression.png', dpi=150)
    plt.show()
    
    print(f"\nNumber of runs: {len(errors)}")
    print(f"Best error: {errors.min():.2f}% at run {runs[errors.argmin()]}")
    print(f"Final error: {errors[-1]:.2f}%")

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

# === FONTSIZE CONFIGURATION ===
BASE_FONTSIZE = 14  # Change this value to adjust all fonts proportionally
TITLE_FONTSIZE = BASE_FONTSIZE
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 2
TICK_FONTSIZE = BASE_FONTSIZE - 2
COLORBAR_FONTSIZE = BASE_FONTSIZE - 2

# Set font properties
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 150

# Predefined colors
colors = ['black', 'red', 'blue', 'green', 'orange', 'cyan']

# === LOAD DATA ===
print("Loading plot replication data...")
data = np.load('../Results/DeepONet_codes/Burgers_Periodic_ICs/2025_11_09_1823_plots/plot_replication_data.npz')

# Extract all arrays
it_log_T = data['it_log_T']
loss_log_T = data['loss_log_T']
train_L2 = data['train_L2']
test_L2 = data['test_L2']
ngsteps_stage1 = data['ngsteps_stage1']
n_time = int(data['n_time'])
n_spatial = int(data['n_spatial'])
mean_error_over_time_train = data['mean_error_over_time_train']
std_error_over_time_train = data['std_error_over_time_train']
mean_error_over_time_test = data['mean_error_over_time_test']
std_error_over_time_test = data['std_error_over_time_test']

# Load prediction data for visualization
S_test = data['S_test']
S_pred_test = data['S_pred_test']

print("✓ Data loaded successfully")

# Parse DeepONet trial history
deeponet_file = '../Results/DeepONet_codes/Burgers_Periodic_ICs/2025_11_09_1823_plots/Experiment_Report.md'
df_deeponet = parse_deeponet_trial_history(deeponet_file)
deeponet_errors = df_deeponet['Error'].values
deeponet_runs = np.arange(len(deeponet_errors))

print(f"✓ Loaded DeepONet results: {len(deeponet_errors)} runs")

# Compute per-sample errors to find best and worst cases
norm_diff = np.linalg.norm(S_test - S_pred_test, axis=1)
norm_truth = np.linalg.norm(S_test, axis=1)
per_sample_error = norm_diff / norm_truth

# Get sorted indices
sorted_indices = np.argsort(per_sample_error)

# Get 2 worst and 2 best
worst_idx_1 = sorted_indices[-1]
worst_idx_2 = sorted_indices[-2]
best_idx_1 = sorted_indices[0]
best_idx_2 = sorted_indices[1]

print(f"Worst sample 1: {worst_idx_1} (Error: {per_sample_error[worst_idx_1]:.4e})")
print(f"Worst sample 2: {worst_idx_2} (Error: {per_sample_error[worst_idx_2]:.4e})")
print(f"Best sample 1: {best_idx_1} (Error: {per_sample_error[best_idx_1]:.4e})")
print(f"Best sample 2: {best_idx_2} (Error: {per_sample_error[best_idx_2]:.4e})")

# Create grid for visualization
x_plot = np.linspace(0, 1, n_spatial)
t_plot = np.linspace(0, 1, n_time)
XX, TT = np.meshgrid(x_plot, t_plot)

# Reconstruct y_all for griddata
T_grid, X_grid = np.meshgrid(np.linspace(0, 1, n_time), np.linspace(0, 1, n_spatial), indexing='ij')
y_all = np.c_[T_grid.flatten(), X_grid.flatten()]

# === CREATE 3x4 PLOT ===
fig, axes = plt.subplots(3, 4, figsize=(12, 7))

# --- ROW 1: ANALYSIS PLOTS ---

# Plot 1: Total Loss
ax = axes[0, 0]
ax.plot(it_log_T, loss_log_T, linewidth=1.5, color=colors[0])
ax.set_title('Loss (MSE)', fontsize=TITLE_FONTSIZE)
ax.set_xlabel('Gradient steps', fontsize=LABEL_FONTSIZE)
ax.set_yscale('log')
ax.axvline(x=ngsteps_stage1, color='r', linestyle='--', linewidth=1, alpha=0.8)
ax.grid(True, alpha=0.4, linestyle='--', which='both')
ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot 2: Train vs Test Error
ax = axes[0, 1]
ax.plot(it_log_T, train_L2, linewidth=1.5, color=colors[0], label='Train')
ax.plot(it_log_T, test_L2, linewidth=1.5, color=colors[4], label='Test')
ax.set_title('Convergence History', fontsize=TITLE_FONTSIZE)
ax.set_xlabel('Gradient steps', fontsize=LABEL_FONTSIZE)
ax.set_ylabel('Relative $L^2$ Error', fontsize=LABEL_FONTSIZE)
ax.set_yscale('log')
ax.axvline(x=ngsteps_stage1, color='r', linestyle='--', linewidth=1, alpha=0.8)
ax.legend(fontsize=LEGEND_FONTSIZE)
ax.grid(True, alpha=0.4, linestyle='--', which='both')
ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot 3: Error Accumulation Over Time
ax = axes[0, 2]
t_plot_axis = np.linspace(0, 1, n_time)

ax.plot(t_plot_axis, mean_error_over_time_train, linewidth=1.5, color=colors[0], label='Train')
ax.fill_between(t_plot_axis, 
                mean_error_over_time_train - std_error_over_time_train,
                mean_error_over_time_train + std_error_over_time_train,
                alpha=0.3, color=colors[0])

ax.plot(t_plot_axis, mean_error_over_time_test, linewidth=1.5, color=colors[4], label='Test')
ax.fill_between(t_plot_axis, 
                mean_error_over_time_test - std_error_over_time_test,
                mean_error_over_time_test + std_error_over_time_test,
                alpha=0.3, color=colors[4])

ax.set_xlabel('Time', fontsize=LABEL_FONTSIZE)
ax.set_title('Accumulation Error', fontsize=TITLE_FONTSIZE)
ax.grid(True, alpha=0.4, linestyle='--', which='both')
ax.set_ylim([1e-2, 1e-1])
ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot 4: DeepONet Error Progression
ax = axes[0, 3]
runs_with_baseline = np.concatenate([[0], deeponet_runs])
errors_with_baseline = np.concatenate([[100], deeponet_errors])

ax.plot(runs_with_baseline, errors_with_baseline/100, 'o-', 
        linewidth=1.5, markersize=5, color=colors[2], alpha=0.8)
ax.set_title('Error Refinement', fontsize=TITLE_FONTSIZE)
ax.set_xlabel('$n$', fontsize=LABEL_FONTSIZE)
ax.set_yscale('log')
ax.grid(True, alpha=0.4, linestyle='--', which='both')
ax.set_ylim([1.5e-2, 2e0])
ax.axhline(y=0.05, color='k', linestyle='--', linewidth=1, alpha=0.5)
ax.tick_params(labelsize=TICK_FONTSIZE)

# --- ROW 2: PREDICTIONS (2 WORST + 2 BEST) ---

sample_indices = [worst_idx_1, worst_idx_2, best_idx_1, best_idx_2]
sample_labels = ['Worst 1', 'Worst 2', 'Best 1', 'Best 2']

predictions = []
errors = []

for idx in sample_indices:
    u_test_sample = S_test[idx]
    u_pred_sample = S_pred_test[idx]
    
    S_pred = griddata(y_all, u_pred_sample.flatten(), (XX, TT), method='cubic')
    S_test_interp = griddata(y_all, u_test_sample.flatten(), (XX, TT), method='cubic')
    pointwise_error = np.abs(S_test_interp - S_pred)
    
    predictions.append(S_pred)
    errors.append(pointwise_error)

# Plot predictions in row 2
for col, (pred, label) in enumerate(zip(predictions, sample_labels)):
    ax = axes[1, col]
    contour = ax.contourf(XX, TT, pred, levels=100, cmap='jet')
    cbar = fig.colorbar(contour, ax=ax)
    cbar.ax.tick_params(labelsize=COLORBAR_FONTSIZE)
    ax.set_title(f'{label} Prediction', fontsize=TITLE_FONTSIZE)
    ax.set_xlabel('$t$', fontsize=LABEL_FONTSIZE)
    if col == 0:
        ax.set_ylabel('$x$', fontsize=LABEL_FONTSIZE)
    ax.set_xticks([])
    ax.set_yticks([])

# --- ROW 3: ERRORS ---

for col, (error, label) in enumerate(zip(errors, sample_labels)):
    ax = axes[2, col]
    contour = ax.contourf(XX, TT, error, levels=100, cmap='jet')
    cbar = fig.colorbar(contour, ax=ax)
    cbar.ax.tick_params(labelsize=COLORBAR_FONTSIZE)
    ax.set_title(f'{label} Error', fontsize=TITLE_FONTSIZE)
    ax.set_xlabel('$t$', fontsize=LABEL_FONTSIZE)
    if col == 0:
        ax.set_ylabel('$x$', fontsize=LABEL_FONTSIZE)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.savefig('deeponet_full_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Summary ===")
print(f"Final train error at t=1: {mean_error_over_time_train[-1]:.4e} ± {std_error_over_time_train[-1]:.4e}")
print(f"Final test error at t=1: {mean_error_over_time_test[-1]:.4e} ± {std_error_over_time_test[-1]:.4e}")
print(f"DeepONet best error: {deeponet_errors.min():.2f}% at run {deeponet_runs[deeponet_errors.argmin()]}")
print(f"DeepONet final error: {deeponet_errors[-1]:.2f}%")
print("\n✓ Saved: deeponet_full_analysis.png")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# ============================================================================
# FONTSIZE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16  # Change this value to adjust all fonts proportionally
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 6
TICK_FONTSIZE = BASE_FONTSIZE - 4

# Set font properties
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# 1x4 PLOT: SciML vs Numerical Methods + Analysis
# ============================================================================

Files_SciML = {
    'AC': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'B': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md'
}

Files_Numerical = {
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2025_12_08_0014/Experiment_Report.md',
    'KH': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'cNS': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}

Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# Parse all sets
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}
dfs_rt = {name: parse_trial_history(path) for name, path in Analysis_RT.items()}
dfs_hm = {name: parse_trial_history(path) for name, path in Analysis_HM.items()}

# Create 1x4 subplot
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

# Predefined colors (max 6 elements)
colors = ['black', 'red', 'blue', 'green', 'orange', 'cyan']

# Function to plot data
def plot_panel(ax, dfs, title, colors,y_label='Reward',x_label='Total Iterations', show_label=False,show_legend=True):
    print(ax)
    for idx, (name, df) in enumerate(dfs.items()):
        runs = [0] + df['Run'].tolist()
        scores = [0] + df['Total'].tolist()
        ax.plot(runs, scores, marker=markers[idx % len(markers)], 
                color=colors[idx % len(colors)],
                linewidth=1.5, markersize=5, label=name, alpha=0.8)
    
    ax.set_xlabel(x_label, fontsize=LABEL_FONTSIZE)
    if show_label:
        ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, ncols=2, loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_ylim([0, 105])
    ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot each panel
plot_panel(axes[0], dfs_numerical, 'SciC (All)', colors,x_label='Total Iterations',y_label='Reward',show_label=True)
plot_panel(axes[1], dfs_sciml, 'SciML (All)', colors,x_label='Total Iterations',y_label='Reward',show_label=True)
plot_panel(axes[2], dfs_rt, 'SciC (RT)', colors,x_label='$T$',y_label='Regret',show_label=True,show_legend=True)
plot_panel(axes[3], dfs_hm, 'SciML (HM)', colors,x_label='$T$',y_label='Regret',show_label=False,show_legend=True)

plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# ============================================================================
# FONTSIZE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16  # Change this value to adjust all fonts proportionally
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 7
TICK_FONTSIZE = BASE_FONTSIZE - 4

# Set font properties
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# 1x4 PLOT: SciML vs Numerical Methods + Analysis
# ============================================================================

Files_SciML = {
    'AC': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'B': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md'
}

Files_Numerical = {
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2025_12_08_0014/Experiment_Report.md',
    'KH': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'cNS': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}

Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# Parse all sets
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}
dfs_rt = {name: parse_trial_history(path) for name, path in Analysis_RT.items()}
dfs_hm = {name: parse_trial_history(path) for name, path in Analysis_HM.items()}

# Create 1x4 subplot
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

# Predefined colors (max 6 elements)
colors = ['black', 'red', 'blue', 'green', 'orange', 'cyan']

# Function to compute regret with padding
def compute_regret(dfs):
    """
    Compute regret: cumulative sum of (100 - average_value_at_T)
    - Ignore first entry (start from index 1)
    - Pad shorter sequences with their last value
    """
    # Extract all score sequences (ignoring first entry which is the baseline 0)
    all_scores = []
    for name, df in dfs.items():
        scores = df['Total'].tolist()  # Don't include the baseline 0
        all_scores.append(scores)
    
    # Find maximum length
    max_len = max(len(scores) for scores in all_scores)
    
    # Pad sequences with last value
    padded_scores = []
    for scores in all_scores:
        if len(scores) < max_len:
            # Pad with last value
            padded = scores + [scores[-1]] * (max_len - len(scores))
        else:
            padded = scores
        padded_scores.append(padded)
    
    # Convert to array and compute average across problems
    padded_scores = np.array(padded_scores)  # Shape: (n_problems, max_len)
    avg_scores = np.mean(padded_scores, axis=0)  # Shape: (max_len,)
    
    # Compute regret: cumulative sum of (100 - avg_score_at_T)
    T = np.arange(1, max_len + 1)  # T starts from 1
    regret = np.cumsum(100 - avg_scores)
    
    return T**2, regret

# Function to plot data
def plot_panel(ax, dfs, title, colors, ncols=1, y_label='Reward', x_label='Total Iterations', show_label=False, show_legend=True, plot_type='reward'):
    print(ax)
    
    if plot_type == 'reward':
        # For REWARD plots: use actual run numbers with lines and markers
        for idx, (name, df) in enumerate(dfs.items()):
            runs = [0] + df['Run'].tolist()
            scores = [0] + df['Total'].tolist()
            ax.plot(runs, scores, marker=markers[idx % len(markers)],
                    color=colors[idx % len(colors)],
                    linewidth=1.5, markersize=5, label=name, alpha=0.8)
        
        if show_label:
            ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
    else:
        # For REGRET plots: compute and plot regret
        T, regret = compute_regret(dfs)
        ax.plot(T, regret, 
                color=colors[0],  # Use single color for average
                linewidth=2, label=r'Regret($T$)', alpha=0.8)
        
        # Always set ylabel for regret plots
        if show_label:
            ax.set_ylabel(r'$\mathbb{E}\left[\sum_{t=1}^T[R^* - R_t]\right]$', fontsize=LABEL_FONTSIZE)
    
    ax.set_xlabel(x_label, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, ncols=ncols, loc='lower right' if plot_type == 'reward' else 'upper left')
    ax.grid(True, alpha=0.3, linestyle='--')
    
    if plot_type == 'reward':
        ax.set_ylim([0, 105])
    
    ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot each panel
# Reward plots: actual run numbers with lines and markers
plot_panel(axes[0], dfs_numerical, 'SciC (All)', colors, x_label='Total Iterations', y_label='Reward ($R$)', show_label=True, plot_type='reward')
plot_panel(axes[1], dfs_sciml, 'SciML (All)', colors, x_label='Total Iterations', ncols=2, y_label='Reward', show_label=False, plot_type='reward')

# Regret plots: computed regret with padding (no dashed line)
plot_panel(axes[2], dfs_rt, 'SciC (RT)', colors, x_label='$T^2$', y_label='Regret', ncols=1, show_label=True, show_legend=False, plot_type='regret')
plot_panel(axes[3], dfs_hm, 'SciML (HM)', colors, x_label='$T^2$', y_label='Regret', ncols=1, show_label=False, show_legend=False, plot_type='regret')

plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy import stats

# ============================================================================
# FONTSIZE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16  # Change this value to adjust all fonts proportionally
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 7
TICK_FONTSIZE = BASE_FONTSIZE - 4

# Set font properties
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# 1x4 PLOT: SciML vs Numerical Methods + Analysis
# ============================================================================

Files_SciML = {
    'AC': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'B': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_ICs/2025_12_05_0057/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md'
}

Files_Numerical = {
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2025_12_08_0014/Experiment_Report.md',
    'KH': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'cNS': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}

Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2123/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# Parse all sets
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}
dfs_rt = {name: parse_trial_history(path) for name, path in Analysis_RT.items()}
dfs_hm = {name: parse_trial_history(path) for name, path in Analysis_HM.items()}

# Create 1x4 subplot
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']

# Predefined colors (max 6 elements)
colors = ['black', 'red', 'blue', 'green', 'orange', 'cyan']

# Function to compute regret with padding
def compute_regret(dfs):
    """
    Compute regret: cumulative sum of (100 - average_value_at_T)
    - Ignore first entry (start from index 1)
    - Pad shorter sequences with their last value
    """
    # Extract all score sequences (ignoring first entry which is the baseline 0)
    all_scores = []
    for name, df in dfs.items():
        scores = df['Total'].tolist()  # Don't include the baseline 0
        all_scores.append(scores)
    
    # Find maximum length
    max_len = max(len(scores) for scores in all_scores)
    
    # Pad sequences with last value
    padded_scores = []
    for scores in all_scores:
        if len(scores) < max_len:
            # Pad with last value
            padded = scores + [scores[-1]] * (max_len - len(scores))
        else:
            padded = scores
        padded_scores.append(padded)
    
    # Convert to array and compute average across problems
    padded_scores = np.array(padded_scores)  # Shape: (n_problems, max_len)
    avg_scores = np.mean(padded_scores, axis=0)  # Shape: (max_len,)
    
    # Compute regret: cumulative sum of (100 - avg_score_at_T)
    T = np.arange(1, max_len + 1)  # T starts from 1
    regret = np.cumsum(100 - avg_scores)
    
    return T**(1/2), regret

# Function to plot data
def plot_panel(ax, dfs, title, colors, ncols=1, y_label='Reward', x_label='Total Iterations', show_label=False, show_legend=True, plot_type='reward'):
    print(ax)
    
    if plot_type == 'reward':
        # For REWARD plots: use actual run numbers with lines and markers
        for idx, (name, df) in enumerate(dfs.items()):
            runs = [0] + df['Run'].tolist()
            scores = [0] + df['Total'].tolist()
            ax.plot(runs, scores, marker=markers[idx % len(markers)],
                    color=colors[idx % len(colors)],
                    linewidth=1.5, markersize=5, label=name, alpha=0.8)
        
        if show_label:
            ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
    else:
        # For REGRET plots: compute and plot regret
        T_squared, regret = compute_regret(dfs)
        ax.plot(T_squared, regret, 
                color=colors[0],  # Use single color for average
                linewidth=2, label=r'$\mathcal{{R}}(T)$', alpha=0.8)
        
        # Fit a straight line: regret = slope * T^2 + intercept
        slope, intercept, r_value, p_value, std_err = stats.linregress(T_squared, regret)
        fit_line = slope * T_squared + intercept
        
        # Plot the fitted line
        ax.plot(T_squared, fit_line, 
                color='gray', linestyle='--', linewidth=2,
                label=f'$\mathcal{{R}}={slope:.2f}\sqrt{{T}}{intercept:+.2f}$', alpha=0.7)
        
        # Always set ylabel for regret plots
        if show_label:
            ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
            #ax.set_ylabel(r'$\mathbb{E}\left[\sum_{t=1}^T[R^* - R_t]\right]$', fontsize=LABEL_FONTSIZE)
    
    ax.set_xlabel(x_label, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, ncols=ncols, loc='lower right')
    ax.grid(True, alpha=0.3, linestyle='--')
    
    if plot_type == 'reward':
        ax.set_ylim([0, 105])
    
    ax.tick_params(labelsize=TICK_FONTSIZE)

# Plot each panel
# Reward plots: actual run numbers with lines and markers
plot_panel(axes[0], dfs_numerical, 'SciC (All)', colors, x_label='Total Iterations', y_label='Reward', show_label=True, plot_type='reward')
plot_panel(axes[1], dfs_sciml, 'SciML (All)', colors, x_label='Total Iterations', ncols=2, y_label='Reward', show_label=False, plot_type='reward')

# Regret plots: computed regret with padding and linear fit
plot_panel(axes[2], dfs_rt, 'SciC (RT)', colors, x_label='$\sqrt{{T}}$', y_label='Regret ($\mathcal{{R}}$)', ncols=1, show_label=True, show_legend=True, plot_type='regret')
plot_panel(axes[3], dfs_hm, 'SciML (HM)', colors, x_label='$\sqrt{{T}}$', y_label='Regret ($\mathcal{{R}}$)', ncols=1, show_label=False, show_legend=True, plot_type='regret')

plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from scipy import stats
import pandas as pd

# ============================================================================
# FONTSIZE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 4
TICK_FONTSIZE = BASE_FONTSIZE - 4

mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# DATASETS
# ============================================================================

Files_SciML_analytic = {
    'BS1D':'../Results/NM_codes/BlackScholes_Dirichlet_CallPayoff/2026_01_11_0059/Experiment_Report.md',
    'VB1D': '../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1605/Experiment_Report.md',
    'DE2D':'../Results/NM_codes/AnisotropicDiffusion_Periodic_SinPlusCos/2026_01_11_0116/Experiment_Report.md',
    'IB2D': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248/Experiment_Report.md',
    'W3D':'../Results/NM_codes/Wave_Neumann_GaussianSin/2026_01_11_1252/Experiment_Report.md',
}

Files_SciML = {
    'A': '../Results/PINN_codes/AllenCahn_Periodic_X2CosPiX/2025_12_05_1208/Experiment_Report.md',
    'VB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md',
    'P':'../Results/PINN_codes/Poisson_Dirichlet_Sin4pixSin4piy/2026_01_10_2319/Experiment_Report.md',
    'H': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'KdV': '../Results/PINN_codes/KdV_Mixed_DerivedIC/2026_01_10_2026/Experiment_Report.md',
    'IB': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'INS': '../Results/PINN_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_0131/Experiment_Report.md',
}

Files_Numerical = {
    'VB': '../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1554/Experiment_Report.md',
    'CH': '../Results/NM_codes/CahnHilliard_Periodic_Noise/2026_01_17_1918/Experiment_Report.md',
    'KH': '../Results/NM_codes/Euler_Periodic_SlipWalls_ShearLayer_ICs/2025_12_06_2249/Experiment_Report.md',
    'RT': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
    'cNS': '../Results/NM_codes/CompressibleNavierStokes_Periodic_NoSlipAdiabatic_SmoothedInterface/2025_12_08_0914/Experiment_Report.md',
    'INS': '../Results/NM_codes/Inverse_NavierStokes_Find_Re_HiddenField_v/2025_11_27_2250/Experiment_Report.md'
}


Analysis_IB2D = {
    'IB1': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248/Experiment_Report.md',
    'IB2': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_12_15_2303/Experiment_Report.md',
    'IB3': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1053/Experiment_Report.md',
    'IB4': '../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/Experiment_Report.md',
    'IB5': '../Results/NM_codes/Burgers_Periodic_SinHalfSum/2025_11_21_0222/Experiment_Report.md',
}

Analysis_RT = {
    'RT1': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_0937/Experiment_Report.md',
    'RT2': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface/2025_12_05_2322/Experiment_Report.md',
    'RT3': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_1059/Experiment_Report.md',
    'RT4': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2025_12_09_2125/Experiment_Report.md',
    'RT5': '../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
}

Analysis_HM = {
    'HM1': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_05_1421/Experiment_Report.md',
    'HM2': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1A2/2025_12_07_1232/Experiment_Report.md',
    'HM3': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1331/Experiment_Report.md',
    'HM4': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1656/Experiment_Report.md',
    'HM5': '../Results/PINN_codes/Helmholtz_Periodic_SinPiA1SinPiA2/2025_12_07_1709/Experiment_Report.md',
}

# ============================================================================
# PARSING & PROCESSING
# ============================================================================

dfs_analytic = {name: parse_trial_history(path) for name, path in Files_SciML_analytic.items()}
dfs_numerical = {name: parse_trial_history(path) for name, path in Files_Numerical.items()}
dfs_sciml = {name: parse_trial_history(path) for name, path in Files_SciML.items()}
dfs_ib2d = {name: parse_trial_history(path) for name, path in Analysis_IB2D.items()}
dfs_rt = {name: parse_trial_history(path) for name, path in Analysis_RT.items()}
dfs_hm = {name: parse_trial_history(path) for name, path in Analysis_HM.items()}

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h']
colors = ['black', 'red', 'blue', 'green', 'orange', 'magenta']

# --- ROBUST REGRET FUNCTION ---
def compute_regret(dfs):
    all_scores = []
    
    for name, df in dfs.items():
        if df is None or df.empty or 'Total' not in df.columns:
            continue
        scores = df['Total'].tolist()
        if not scores: continue
        all_scores.append(scores)
    
    if not all_scores:
        return np.array([]), np.array([])
    
    max_len = max(len(s) for s in all_scores)
    padded_scores = []
    for scores in all_scores:
        if len(scores) < max_len:
            padded = scores + [scores[-1]] * (max_len - len(scores))
        else:
            padded = scores
        padded_scores.append(padded)
    
    padded_scores = np.array(padded_scores, dtype=float)
    
    # Use nanmean to safely average even if some runs failed (NaN)
    avg_scores = np.nanmean(padded_scores, axis=0)
    
    if np.isnan(avg_scores).any():
        df_avg = pd.Series(avg_scores)
        avg_scores = df_avg.ffill().fillna(0).values

    T = np.arange(1, max_len + 1)
    regret = np.cumsum(100 - avg_scores)
    
    return (T**(1/2)), regret

# ============================================================================
# FIXED PLOTTING FUNCTION (Handles Single-Point Runs)
# ============================================================================

def plot_panel(ax, dfs, title, colors, ncols=1, y_label='Reward', x_label='Total Iterations', show_label=False, show_legend=True, plot_type='reward'):
    print(f"Plotting {title}...")
    
    if plot_type == 'reward':
        for idx, (name, df) in enumerate(dfs.items()):
            if df is None or df.empty: continue
            runs = [0] + df['Run'].tolist()
            scores = [0] + df['Total'].tolist()
            ax.plot(runs, scores, marker=markers[idx % len(markers)],
                    color=colors[idx % len(colors)],
                    linewidth=1.5, markersize=5, label=name, alpha=0.8)
        
        if show_label:
            ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)

    else:
        # Calculate Regret
        T_sqrt, regret = compute_regret(dfs)
        
        if len(T_sqrt) > 0:
            # --- FIX 1: Add marker='o' ---
            # This ensures that if you only have 1 point (T=1), it still appears as a dot.
            ax.plot(T_sqrt, regret, color=colors[0], linewidth=2, 
                    marker='o', markersize=6, # <--- Added Marker
                    label=r'$\mathcal{{R}}(T)$', alpha=0.8)
            
            # --- Perfect Solver / Single Point Check ---
            if len(T_sqrt) == 1:
                print(f"  -> Single data point detected for {title}. Skipping regression.")
                # No regression possible with 1 point.
                # Just adjust axes so it looks nice
                ax.set_xlim(0, 2) 
            
            elif np.std(regret) < 1e-9: 
                # Perfect solver (Regret is all zeros)
                intercept = regret[0]
                fit_line = np.full_like(T_sqrt, intercept)
                ax.plot(T_sqrt, fit_line, color='gray', linestyle='--', linewidth=2,
                        label=r'$\mathcal{{R}} \approx 0$', alpha=0.7)
            else:
                # Normal regression
                mask = np.isfinite(T_sqrt) & np.isfinite(regret)
                if np.sum(mask) > 1:
                    slope, intercept, r_value, p_value, std_err = stats.linregress(T_sqrt[mask], regret[mask])
                    fit_line = slope * T_sqrt + intercept
                    ax.plot(T_sqrt, fit_line, color='gray', linestyle='--', linewidth=2,
                            label=f'$\mathcal{{R}}={slope:.2f}\sqrt{{T}}{intercept:+.2f}$', alpha=0.7)
        
        if show_label:
            ax.set_ylabel(y_label, fontsize=LABEL_FONTSIZE)
    
    ax.set_xlabel(x_label, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    
    # Legend location fix
    if show_legend:
        # If there is very little data (like the single point), move legend to avoid covering it
        loc = 'upper right' if plot_type == 'regret' and len(T_sqrt) < 3 else 'lower right'
        ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, ncols=ncols, loc=loc)
        
    ax.grid(True, alpha=0.3, linestyle='--')
    
    if plot_type == 'reward':
        ax.set_ylim([0, 105])
    
    ax.tick_params(labelsize=TICK_FONTSIZE)

# --- EXECUTION ---
# Row 1
plot_panel(axes[0, 0], dfs_analytic, 'SciC (Analytical)', colors, x_label='Total Iterations', y_label='Reward', show_label=True, plot_type='reward')
plot_panel(axes[0, 1], dfs_numerical, 'SciC (Numerical)', colors, x_label='Total Iterations', y_label='Reward', show_label=False, plot_type='reward')
plot_panel(axes[0, 2], dfs_sciml, 'SciML', colors, x_label='Total Iterations', y_label='Reward', show_label=False, ncols=2, plot_type='reward')

# Row 2
plot_panel(axes[1, 0], dfs_ib2d, 'SciC (IB2D) ', colors, x_label='$\sqrt{{T}}$', y_label='Regret ($\mathcal{{R}}$)', ncols=1, show_label=True, show_legend=True, plot_type='regret')
plot_panel(axes[1, 1], dfs_rt, 'SciC (RT)', colors, x_label='$\sqrt{{T}}$', y_label='Regret ($\mathcal{{R}}$)', ncols=1, show_label=False, show_legend=True, plot_type='regret')
plot_panel(axes[1, 2], dfs_hm, 'SciML (H) ', colors, x_label='$\sqrt{{T}}$', y_label='Regret ($\mathcal{{R}}$)', ncols=1, show_label=False, show_legend=True, plot_type='regret')

plt.tight_layout()
plt.savefig('athena_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import re
import pandas as pd
from pathlib import Path

def parse_trial_history(file_path):
    """
    Parse the Full Trial History table from an Experiment_Report.md file.
    
    Parameters
    ----------
    file_path : str or Path
        Path to the Experiment_Report.md file
        
    Returns
    -------
    pd.DataFrame
        DataFrame with columns: Run, Model Description, Error, Integrity, 
        Accuracy, Details, Optimality, Total
        
    Examples
    --------
    >>> Files = {
    ...     'Burgers': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_05_1414/Experiment_Report.md'
    ... }
    >>> df = parse_trial_history(Files['Burgers'])
    >>> # Plot error vs run
    >>> import matplotlib.pyplot as plt
    >>> plt.semilogy(df['Run'], df['Error'])
    >>> plt.xlabel('Run')
    >>> plt.ylabel('Error')
    >>> # Extract scores
    >>> print(df[['Run', 'Total']])
    """
    with open(file_path, 'r') as f:
        content = f.read()
    
    # Find the Full Trial History section
    history_pattern = r'## Full Trial History\s*\n(.*?)(?=\n##|\n###|$)'
    match = re.search(history_pattern, content, re.DOTALL)
    
    if not match:
        raise ValueError("Could not find 'Full Trial History' section in file")
    
    history_section = match.group(1)
    
    # Split into lines and find the table
    lines = history_section.strip().split('\n')
    
    # Find header and separator lines
    header_idx = None
    sep_idx = None
    for i, line in enumerate(lines):
        if '| Run |' in line or '| :--- |' in line:
            if header_idx is None and '| Run |' in line:
                header_idx = i
            elif sep_idx is None and '| :--- |' in line:
                sep_idx = i
                break
    
    if header_idx is None or sep_idx is None:
        raise ValueError("Could not find table header in Full Trial History")
    
    # Parse header
    header_line = lines[header_idx]
    headers = [h.strip().replace('**', '') for h in header_line.split('|')[1:-1]]
    
    # Parse data rows (everything after separator)
    data_rows = []
    for line in lines[sep_idx + 1:]:
        line = line.strip()
        if not line or not line.startswith('|'):
            break
        
        # Split by | and clean up, removing markdown bold formatting
        cells = [cell.strip().replace('**', '') for cell in line.split('|')[1:-1]]
        if len(cells) == len(headers):
            data_rows.append(cells)
    
    # Create DataFrame
    df = pd.DataFrame(data_rows, columns=headers)
    
    # Convert numeric columns
    df['Run'] = df['Run'].astype(int) + 1  # Add 1 to all run numbers since files start at 0
    
    # Convert Error column (handles scientific notation like 1.00e-06, percentages like 0.02%, and N/A)
    def parse_error(x):
        x = x.strip()
        if not x or x.upper() == 'N/A' or x.lower() == 'not applicable':
            return 0.0
        # Handle percentage values
        if '%' in x:
            return float(x.replace('%', '')) / 100.0
        # Handle scientific notation and regular floats
        return float(x)
    
    df['Error'] = df['Error'].apply(parse_error)
    
    # Convert score columns
    score_cols = ['Integrity', 'Accuracy', 'Details', 'Optimality', 'Total']
    for col in score_cols:
        df[col] = df[col].astype(int)
    
    return df


def extract_metrics(df, metric_col):
    """
    Extract a specific metric column for easy plotting.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame returned by parse_trial_history
    metric_col : str
        Column name to extract (e.g., 'Error', 'Total', 'Accuracy')
        
    Returns
    -------
    runs : array
        Run numbers
    values : array
        Metric values
    """
    return df['Run'].values, df[metric_col].values


def plot_error_progression(df, ax=None):
    """
    Quick plot of error vs run number.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame from parse_trial_history
    ax : matplotlib axis, optional
        Axis to plot on. If None, creates new figure.
        
    Returns
    -------
    ax : matplotlib axis
    """
    import matplotlib.pyplot as plt
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.semilogy(df['Run'], df['Error'], 'o-', linewidth=2, markersize=8)
    ax.set_xlabel('Run', fontsize=12)
    ax.set_ylabel('Relative L2 Error', fontsize=12)
    ax.set_title('Error Progression Across Runs', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    return ax


def plot_score_breakdown(df, ax=None):
    """
    Stacked bar chart showing score breakdown by component.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame from parse_trial_history
    ax : matplotlib axis, optional
        Axis to plot on. If None, creates new figure.
        
    Returns
    -------
    ax : matplotlib axis
    """
    import matplotlib.pyplot as plt
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 6))
    
    score_cols = ['Integrity', 'Accuracy', 'Details', 'Optimality']
    
    bottom = None
    for col in score_cols:
        if bottom is None:
            ax.bar(df['Run'], df[col], label=col)
            bottom = df[col].values
        else:
            ax.bar(df['Run'], df[col], bottom=bottom, label=col)
            bottom = bottom + df[col].values
    
    ax.set_xlabel('Run', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Score Breakdown Across Runs', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    return ax

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ============================================================================
# 1. FONTSIZE & STYLE CONFIGURATION
# ============================================================================
BASE_FONTSIZE = 16
TITLE_FONTSIZE = BASE_FONTSIZE + 2
LABEL_FONTSIZE = BASE_FONTSIZE
LEGEND_FONTSIZE = BASE_FONTSIZE - 4
TICK_FONTSIZE = BASE_FONTSIZE - 2

mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = BASE_FONTSIZE
mpl.rcParams['figure.dpi'] = 100

# ============================================================================
# 2. DATASETS
# ============================================================================

Files_SciC_IB = {
    'ATHENA': '../Results/NM_codes/Burgers_Periodic_SinSum/2025_12_15_2248/Experiment_Report.md',
    'No Scaffolding': '../Results/NM_codes/Burgers_Periodic_SinSum/2026_01_10_1309/Experiment_Report_fix_table.md',
}

Files_SciC_RT = {
    'ATHENA':'../Results/NM_codes/CompressibleEuler_Periodic_Reflective_SmoothedInterface_ICs/2025_12_06_2223/Experiment_Report.md',
    'No Scaffolding': '../Results/NM_codes/CompressibleEuler_PeriodicReflective_SmoothedInterface/2026_01_10_1232/Experiment_Report.md',
}

Files_SciML_Kdv = {
    'ATHENA': '../Results/PINN_codes/KdV_Mixed_DerivedIC/2026_01_10_2026/Experiment_Report.md',
    'No Scaffolding': '../Results/PINN_codes/KdV_Mixed_AnalyticalIC/2026_01_09_1804/Experiment_Report.md',
}

Files_SciML_IB = {
    'ATHENA': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2025_12_04_1621/Experiment_Report.md',
    'No Scaffolding': '../Results/PINN_codes/Burgers_Periodic_SinPiX/2026_01_10_1758/Experiment_Report.md',
}

# ============================================================================
# 3. LOAD DATA (Using existing parser)
# ============================================================================

print("Loading Data using existing parser...")
dfs_scic_ib = {k: parse_trial_history(v) for k, v in Files_SciC_IB.items()}
dfs_scic_rt = {k: parse_trial_history(v) for k, v in Files_SciC_RT.items()}
dfs_sciml_kdv = {k: parse_trial_history(v) for k, v in Files_SciML_Kdv.items()}
dfs_sciml_ib = {k: parse_trial_history(v) for k, v in Files_SciML_IB.items()}

# ============================================================================
# 4. PLOTTING
# ============================================================================

# Adjusted figsize for 1x4 layout
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

# Style settings
colors_map = {'ATHENA': 'black', 'No Scaffolding': 'red'} 
markers_map = {'ATHENA': 'o', 'No Scaffolding': 's'}

linestyles_map = {'ATHENA': '-', 'No Scaffolding': '--'}

def plot_reward_panel(ax, data_dict, title, y_label=False, show_legend=False):
    ax.set_title(title, fontsize=TITLE_FONTSIZE, pad=10)
    
    for name, df in data_dict.items():
        if df is None or df.empty:
            print(f"  -> No data found for {name} in {title}")
            continue
            
        # Ensure we are plotting Run vs Total
        if 'Run' in df.columns and 'Total' in df.columns:
            # Prepend 0 to runs and scores
            runs = [0] + df['Run'].tolist()
            scores = [0] + df['Total'].tolist()
            
            ax.plot(runs, scores, 
                    label=name,
                    color=colors_map.get(name, 'black'),
                    marker=markers_map.get(name, 'o'),
                    linestyle=linestyles_map.get(name, '-'),
                    linewidth=2, 
                    markersize=6, 
                    alpha=0.8)

    ax.set_xlabel("Total Iterations", fontsize=LABEL_FONTSIZE)
    if y_label:
        ax.set_ylabel("Reward", fontsize=LABEL_FONTSIZE)
    
    # Standardize y-axis for 0-100 scores
    ax.set_ylim(-5, 105) 
    ax.grid(True, which='both', linestyle='--', alpha=0.5)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, framealpha=0.95, loc='lower right')

# --- Plot 1x4 Layout ---
# Only show legend and y-label on the first plot (index 0)
plot_reward_panel(axes[0], dfs_scic_ib, "SciC (IB2D)", y_label=True, show_legend=True)
plot_reward_panel(axes[1], dfs_scic_rt, "SciC (RT)", y_label=False, show_legend=False)
plot_reward_panel(axes[2], dfs_sciml_kdv, "SciML (KdV)", y_label=False, show_legend=False)
plot_reward_panel(axes[3], dfs_sciml_ib, "SciML (IB)", y_label=False, show_legend=False)

plt.tight_layout()
filename = "ablation_study_reward_comparison_1x4.png"
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Comparison plot saved to {filename}")